# literature review before continue

## prep the dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import h5py
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../../src')
from utils import smooth_curve
from Viz import show_images
from PlumeDataset import plume_dataset
from AutoAlign import align_plumes

In [6]:
plume_ds_2_25 = plume_dataset(file_path='../../datasets/PlumeMapping_2.25_YichenGuo_04112024.h5', group_name='PLD_Plumes')
coords_standard = np.array([[25,64], [27,199], [375,37], [375,220]])
plume_ds_2 = plume_dataset(file_path='../../datasets/PlumeMapping_2_YichenGuo_04112024.h5', group_name='PLD_Plumes')
coords_2 = np.array([[25,64], [27,199], [375,37], [375,220]])
plume_ds_1_75 = plume_dataset(file_path='../../datasets/PlumeMapping_1.75_YichenGuo_04122024.h5', group_name='PLD_Plumes')
coords_1_75 = np.array([[30,60], [31,193], [374,42], [375,226]])
plume_ds_1_5 = plume_dataset(file_path='../../datasets/PlumeMapping_1.5_YichenGuo_04122024.h5', group_name='PLD_Plumes')
coords_1_5 = np.array([[30,60], [31,193], [374,42], [375,226]])
plume_ds_1_25 = plume_dataset(file_path='../../datasets/PlumeMapping_1.25_YichenGuo_04122024.h5', group_name='PLD_Plumes')
coords_1_25 = np.array([[30,60], [31,193], [374,42], [375,226]])

# analysis frame parameters
coors_standard = np.array([[25,64], [27,199], [375,37], [375,220]])

In [14]:
names = ['1-BaTiO3', '2-BaTiO3', '3-BaTiO3', '4-BaTiO3', '5-BaTiO3', '6-BaTiO3', '7-BaTiO3', '8-BaTiO3', '9-BaTiO3']
pressures = [140, 130, 120, 110, 100, 90, 80, 70, 60]
pressures = [f'{p}mTorr' for p in pressures]
energies = [2.25, 2, 1.75, 1.5, 1.25]
datasets = [plume_ds_2_25, plume_ds_2, plume_ds_1_75, plume_ds_1_5, plume_ds_1_25]
coords = [coords_standard, coords_2, coords_1_75, coords_1_5, coords_1_25]

plumes_all = []
label_energy = []
label_pressure = []
for i, (dataset, coord, energy) in enumerate(zip(datasets, coords, energies)):
    for n, p in zip(names, pressures):
        plumes = dataset.load_plumes(n)
        plumes = align_plumes(plumes, coord, coords_standard)
        plumes_all.append(plumes)
        label_energy.append(str(energy))
        label_pressure.append(p)

plumes_all = np.concatenate(plumes_all, axis=0)
label_energy = np.array(label_energy)
label_pressure = np.array(label_pressure)
plumes_all.shape, label_energy.shape, label_pressure.shape

((899, 128, 250, 400), (45,), (45,))

In [ ]:
with h5py.File('../../datasets/PlumeMapping_dataset.h5', 'a') as h5:
    h5.create_dataset('data', data=plumes_all)
    # h5.create_dataset('energy', data=label_energy)
    # h5.create_dataset('pressure', data=label_pressure)

In [19]:
names = ['1-BaTiO3', '2-BaTiO3', '3-BaTiO3', '4-BaTiO3', '5-BaTiO3', '6-BaTiO3', '7-BaTiO3', '8-BaTiO3', '9-BaTiO3']
pressures = [140, 130, 120, 110, 100, 90, 80, 70, 60]
pressures = [p for p in pressures]
energies = [2.25, 2, 1.75, 1.5, 1.25]
datasets = [plume_ds_2_25, plume_ds_2, plume_ds_1_75, plume_ds_1_5, plume_ds_1_25]
coords = [coords_standard, coords_2, coords_1_75, coords_1_5, coords_1_25]

label_energy = []
label_pressure = []
for i, (dataset, coord, energy) in enumerate(zip(datasets, coords, energies)):
    for n, p in zip(names, pressures):
        plumes = dataset.load_plumes(n)
        label_energy.append([energy]*len(plumes))
        label_pressure.append([p]*len(plumes))

label_energy = np.concatenate(label_energy)
label_pressure = np.concatenate(label_pressure)
label_energy.shape, label_pressure.shape

((899,), (899,))

In [20]:
with h5py.File('../../datasets/PlumeMapping_dataset.h5', 'a') as h5:
    # h5.create_dataset('data', data=plumes_all)
    h5.create_dataset('energy', data=label_energy)
    h5.create_dataset('pressure', data=label_pressure)

## build model

In [7]:
%load_ext autoreload
%autoreload 2

import cv2
import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import sys
sys.path.append('../../src')
from Viz import show_images

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
### h5py datasets for training
class hdf5_dataset(Dataset):
    
    def __init__(self, file_path, transform=None, data_key='data', label_key='labels'):
        self.file_path = file_path
        self.transform = transform
        self.hf = None
        self.data_key = data_key
        self.label_key = label_key

    def __len__(self):
        with h5py.File(self.file_path, 'r') as f:
            self.len = len(f[self.label_key])
        return self.len
    
    def __getitem__(self, idx):
        if self.hf is None:
            self.hf = h5py.File(self.file_path, 'r')
            
        image = np.array(self.hf[self.data_key][idx])
        label = np.array(self.hf[self.label_key][idx])
        
        if self.transform:
            image = self.transform(image)
        return image, label


### packed visualization functions
def viz_dataloader(dl, n=8, title=None, hist_bins=None, show_colorbar=False, label_converter=None, stacked=False):
    batch = next(iter(dl))
    if len(batch[0]) < n: 
        raise ValueError("n is smaller than batch size, increase n")

    inputs = batch[0][:n]
    labels = list(batch[1][:n].numpy())
    if label_converter:
        for i in range(len(labels)):
            labels[i] = label_converter[labels[i]]
    show_images(torch.permute(inputs, [0,2,3,1]).cpu().numpy(), labels=labels, title=title, hist_bins=hist_bins, show_colorbar=show_colorbar) 


bs = 256
# imagenet
imagenet_ds = hdf5_dataset('../../datasets/PlumeMapping_dataset.h5', transform=transforms.ToTensor())
train_dl = DataLoader(imagenet_ds, batch_size=bs, shuffle=True, num_workers=4)
viz_dataloader(train_dl, label_converter=label_converter, title='imagenet - train')

In [4]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()  # Use sigmoid if input is normalized between 0 and 1
        )
        
        # Physical properties predictor
        num_physical_properties = 2
        self.physical_predictor = nn.Linear(64, num_physical_properties)  # adjust size accordingly

    def forward(self, x):
        x = self.encoder(x)
        embedding = x.view(x.size(0), -1)
        physical_properties = self.physical_predictor(embedding)
        x = self.decoder(x)
        return x, physical_properties

def physical_properties_loss(predicted, actual):
    return nn.MSELoss()(predicted, actual)

def total_loss(reconstruction, original, predicted_properties, actual_properties):
    reconstruction_loss = nn.MSELoss()(reconstruction, original)
    # properties_loss = physical_properties_loss(predicted_properties, actual_properties)
    # return reconstruction_loss + properties_loss
    return reconstruction_loss